<!-- GENERATED by scripts/build_problems.py from
     DA4CHE-admin/problems/Topic6.1-Time_Series_Basics/master.md
     Do not edit this file directly — your changes will be overwritten.
     Solutions and rubrics live in the private repo and must never appear here. -->

```{contents}
:local:
:depth: 2
```

# Problems: Time Series Basics

:::{admonition} Get this problem set
:class: seealso

{download}`Download everything (Topic6.1-Time_Series_Basics_Problems.zip) <archives/Topic6.1-Time_Series_Basics_Problems.zip>` — the notebook and
`boiler_historian.csv`, in a folder that is ready to run as-is.

The Download badge at the top of the page will also give you the notebook on its own.
:::

:::{admonition} Before you start
:class: tip

This problem set accompanies {doc}`Time Series Basics </6-advanced_topics/Topic6.1-Time_Series_Basics>`. It is worth **100 points**:

- **Part A — Skill Checks (30 pts)** — short answers, each with a `check` that tells you
  whether it is right; run it as often as you like.
- **Part B — Visualization (35 pts)** — plots plus written interpretation.
- **Part C — Open Ended (35 pts)** — one synthesis problem.

The parts build on each other: Part A works out the syntax you need for Part B, and
Part B produces the evidence you argue from in Part C. Do them in order.
:::

## Setup

In this problem set we will work with real data from the control system of a coal-fired
boiler at a chemical plant in China, published by Hu et al. (2025). The plant's
distributed control system (DCS) saves every sensor reading, or "tag", in a database
called a **data historian**. An engineer exported one day of it, 28 March 2022, at the
historian's 5 s interval, so `data/boiler_historian.csv` has **17,280 rows** of eight
tags:

| column | tag | description |
|---|---|---|
| `furnace_pressure` | PT_8313A | upper furnace pressure (a negative draft) |
| `furnace_temp` | TE_8313B | temperature in the upper furnace chamber |
| `steam_temp` | TE_8332A | boiler outlet steam temperature, °C |
| `primary_air_flow` | FT_8301 | primary fan outlet flow rate |
| `flue_o2` | AIR_8301A | flue gas oxygen at the economizer inlet |
| `drum_pressure` | PTCA_8322A | steam drum pressure |
| `main_steam_flow` | ZZQBCHLL | main steam flow |
| `desuperheat_water_flow` | YJJWSLL | desuperheating water flow |

Note that the paper only gives units for the steam temperature, which normally stays
between 530 and 545 °C, so we will use the other tags as they are stored. Every value has
**two decimal places**, and otherwise the file is exactly what the historian exported,
including a real gap in one of the tags.

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf, adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
try:
    plt.style.use('../settings/plot_style.mplstyle')   # available inside the book
except OSError:
    pass                                              # downloaded notebook: use defaults

df = pd.read_csv('data/boiler_historian.csv', parse_dates=['time'], index_col='time')
print(df.shape)
df.head()

(17280, 8)


,furnace_pressure,furnace_temp,steam_temp,primary_air_flow,flue_o2,drum_pressure,main_steam_flow,desuperheat_water_flow
time,,,,,,,,
2022-03-28 00:00:04,-201.62,775.00,541.46,45040.86,2.33,9.87,60.59,2.83
2022-03-28 00:00:09,-189.87,775.03,541.40,45320.32,2.35,9.86,60.59,2.83
2022-03-28 00:00:14,-196.67,774.93,541.40,45359.98,2.38,9.86,60.37,2.82
2022-03-28 00:00:19,-188.02,774.63,541.34,46088.86,2.38,9.87,60.37,2.82
2022-03-28 00:00:24,-201.37,774.17,541.27,44937.00,2.40,9.87,61.01,2.82


Run this once. It defines the `check` helper used after each Part A answer. It will
not run inside the book — use the downloaded notebook.

In [ ]:
# Run this once. It defines `check`, used after each Part A answer below.
import hashlib

_EXPECTED = {
    'q1': (3, ['08788f53c263', '46926fb6df4c', 'cc62c9fbe9ee', '7cd12e0af9eb', '98ae7e1a1c96', '96f40cdb4921', '81068923bebf']),
    'q2': (2, ['a9f63b26ad16', '2df00558be66', 'a011be0d7b1b', 'bdd71b8165a1', 'e546b5c35bb2', '27b4cf12b9b5', '167bbfae6488', 'f7b9ef9470a7', 'dfd0c7dd25a7', 'a91ac615b7cd', '8cf1190cb6d8']),
    'q3': (4, ['5933d2a95aae', 'be75d9a3e7ec', 'e4003449cc94', '6d007932096a', '180e6f44c027', 'be607312f8ae', '4e32151b4fab', '712f11e87620', '9b5d8901b873', '97e666d87405', 'd9ceb37fb904', '0599527ed58b', '456fa3bce642', 'b66b38208299', '2df492a3062b']),
}

def check(qid, value):
    """Compare an answer against a stored digest. Practice only -- nothing is recorded."""
    places, digests = _EXPECTED[qid]
    if value is Ellipsis:
        print(f"{qid}: not answered yet")
        return
    got = hashlib.sha256(f"{float(value):.{places}f}".encode()).hexdigest()[:12]
    print(f"{qid}: {'correct' if got in digests else 'not the expected value'}")

---

## Part A — Skill Checks (30 pts)

We will start with some basic questions that we should always ask before building a
model from time series data. First, how much of the signal is fast noise that a moving
average would remove? Second, how long does the signal "remember" its past values? Third,
if some data are missing, how well can we fill them in? Each question below looks at one
of these for one of the tags.

### A1. How much of the furnace pressure is fast fluctuation? (10 pts)

:::{exercise}
:label: pr-adv-ts-ma-resid

A moving average splits a signal into two parts. The average itself keeps the slow
changes. What is left over, the **residual** (raw signal minus the average), holds the
changes that are faster than the window. Comparing the residual's variance with the raw
signal's variance tells us what fraction of the signal's variability is fast fluctuation.
A value near 0 means the signal is mostly slow drift, and a value near 1 means it is mostly
fast noise.

1. Smooth `furnace_pressure` with a **trailing** 5-minute moving average,
   `.rolling(60).mean()` (60 samples at 5 s each). Trailing means each average uses only the
   current and past samples, which is the default for `.rolling`.
2. Compute the residual, `furnace_pressure - moving_average`. Its first 59 values are NaN,
   which `.var()` skips.
3. Divide the residual's variance by the raw signal's variance,
   `residual.var() / furnace_pressure.var()`.

Assign that fraction to `frac_resid`.
:::

In [ ]:
# YOUR CODE HERE
frac_resid = ...

In [ ]:
check("q1", frac_resid)

### A2. How long does the furnace temperature remember? (10 pts)

:::{exercise}
:label: pr-adv-ts-acf-half

Compute the autocorrelation function of `furnace_temp` with `acf(..., nlags=720)` (one hour
of lags). Find the **first lag at which the ACF falls below 0.5**.

Assign that lag, **in minutes**, to `lag_half_min`.
:::

In [ ]:
# YOUR CODE HERE
lag_half_min = ...

In [ ]:
check("q2", lag_half_min)

### A3. Filling a gap (10 pts)

The `desuperheat_water_flow` tag has a real 2.3-minute gap at 06:22. We would like to know
how well linear interpolation fills a gap like that, but we cannot check a fill of a real
gap, because the true values are missing. The standard trick is to **make artificial gaps
in data we do have**, fill them, and compare the fill with the values we removed.

Here we make gaps about as long as the real one: 2 minutes, which is 24 samples at 5 s each.
A single gap could land on a quiet stretch or a busy one by luck, so we make six of them,
one every 4 hours, and average the error over all six. The data start at midnight and there
are 720 samples per hour, so the gap starting at hour $h$ begins at row $720h$.

:::{exercise}
:label: pr-adv-ts-interp-mae

1. Make a copy of `steam_temp`, so the original keeps the true values.
2. In the copy, set 24 consecutive samples to `np.nan` starting at each of these row
   positions (02:00, 06:00, 10:00, 14:00, 18:00 and 22:00):

   ```
   1440, 4320, 7200, 10080, 12960, 15840
   ```

   That is 6 × 24 = 144 blanked samples. Use `.iloc` with row positions, not times.
3. Fill the gaps with `.interpolate(method='linear')`.
4. Compute the mean absolute error between the filled values and the true values at the
   **144 blanked positions only**. Everywhere else the copy is identical to the original, so
   including those rows would make the error look tiny.

Assign that error, in °C, to `mae_linear`.
:::

In [ ]:
# YOUR CODE HERE
mae_linear = ...

In [ ]:
check("q3", mae_linear)

:::{admonition} Checkpoint
:class: note

You now have the three tools Part B uses: a trailing moving average, the first lag where the
ACF drops below 0.5, and a measure of how well a fill reproduces known values.
:::

---

## Part B — Visualization (35 pts)

A1 and A2 described one tag each with one number. Part B looks at the whole day, and asks
how the eight tags differ in their noise, their memory and their stationarity. These are the
properties you would check before choosing a smoothing window or a model.

::::{exercise}
:label: pr-adv-ts-boiler-day

**1. Smoothing.** Plot `furnace_pressure` for the full day and, in a second panel, for one
hour (10:00 to 11:00). In both panels overlay the raw signal, the trailing 5-minute moving
average from A1, and an exponential moving average with `ewm(span=60)`. In the full-day panel,
add the 30-minute rolling standard deviation, `.rolling(360).std()`, on a twin y-axis.
`span=60` gives the `ewm` the same average delay as the moving average (2.5 minutes). In
**one sentence**, say which of the two follows the raw signal more closely in the one-hour
panel, and why.

**2. Memory.** For `furnace_pressure` and `furnace_temp`, plot the ACF out to 30 minutes
(`nlags=360`) and the PACF out to 60 lags, in a 2 × 2 grid. For each tag, count the PACF
lags outside the 95% band and report the size of the largest PACF value beyond lag 1. One
tag has more lags outside the band, and the other has larger PACF values. In **one
sentence**, say which tag needs more past values to describe its present value, and why the
count is misleading here.

**3. Stationarity.** Use the Augmented Dickey-Fuller (ADF) test from the chapter
(`adfuller`) to make a table of the p-value for every tag, raw and after first differencing
(drop the NaNs in `desuperheat_water_flow` first). Then plot `steam_temp` for the day with
its 530–545 °C band shaded and its 30-minute rolling mean and rolling standard deviation
overlaid, and report the fraction of samples outside the band. The ADF test calls the steam
temperature stationary. In **one sentence**, use your plot to say what this does and does
not tell us about the signal.

Label all axes, with units where there are any.
::::

In [ ]:
# YOUR CODE HERE

---

## Part C — Open Ended (35 pts)

One thing that is easy to forget is that the numbers in this file are not exactly what
the sensors measured. The historian rounded each value to two decimals, and many plant
historians also compress the data with the swinging-door algorithm from the chapter. This
does not change a trend plot very much, but it can change the statistics we computed in
Part B. In Part C we will see how much, and how fine the historian's settings need to be
before we can trust those statistics.

Use the chapter's swinging-door function, given here:

In [2]:
def swinging_door(t, y, delta):
    """Swinging-door (SDT) compression: indices of the points a historian keeps."""
    keep = [0]
    t0, y0 = t[0], y[0]
    s_max, s_min = -np.inf, np.inf          # lower-door / upper-door slopes
    for i in range(1, len(y)):
        dt = t[i] - t0
        s_min = min(s_min, (y[i] + delta - y0) / dt)
        s_max = max(s_max, (y[i] - delta - y0) / dt)
        if s_max > s_min:                   # corridor closed: archive previous point
            keep.append(i - 1)
            t0, y0 = t[i - 1], y[i - 1]
            dt = t[i] - t0
            s_min = (y[i] + delta - y0) / dt
            s_max = (y[i] - delta - y0) / dt
    keep.append(len(y) - 1)
    return np.array(sorted(set(keep)))

::::{exercise}
:label: pr-adv-ts-historian-audit

**1. Rounding.** The historian stored every value to two decimals, so the true readings
are gone and we cannot undo the rounding. We can still measure how much noise it adds.
Rounding to two decimals moves each value by a random amount between −0.005 and +0.005, so
simulate that error with `e = np.random.default_rng(0).uniform(-0.005, 0.005, len(df))`
and compute the variance of its first difference, `np.diff(e).var()`. This is how much
the rounding alone adds to the variance of a tag's sample-to-sample changes. For every tag,
divide it by the variance of the tag's own first difference, and also compute the fraction
of consecutive samples that are unchanged. Identify the tag for which rounding matters,
and plot 15 minutes of it. In **one sentence**, say which statistic of that tag you would
still trust and which you would not.

**2. Compression.** Compress `steam_temp` with `swinging_door` at
$\delta = 0.01, 0.02, 0.05, 0.1, 0.25, 0.5$ times its standard deviation, and rebuild each
compressed signal with `np.interp`. For each δ, record the compression ratio, the maximum
reconstruction error, the first lag (minutes) at which the ACF falls below 0.5 (A2's
statistic), and the lag-1 ACF of the first difference. Plot all four against δ.

**3. Recommend.** In **one or two sentences**, tell the plant's data engineer what δ to use
for this tag if analysts will rely on its ACF. Use the largest δ on your grid that keeps the
ACF-0.5 lag within 10% of the uncompressed value, give its compression ratio, and say which
statistic is distorted even at that δ. Use your numbers.

There is more than one defensible way to present the trade-off. You are graded on whether
your numbers support your recommendation.
::::

In [ ]:
# YOUR CODE HERE

---


## Additional Reading

1. W. Hu, A. Jiang, K. Chen, J. Zheng, W. Shang and Z. Cao, "A long-tailed distribution
   time-series dataset in boiler equipment," *Scientific Data* **12**, 742 (2025),
   [doi:10.1038/s41597-025-05096-4](https://doi.org/10.1038/s41597-025-05096-4) — the
   source of this dataset; the data record is
   [doi:10.6084/m9.figshare.28868849](https://doi.org/10.6084/m9.figshare.28868849) (CC0).
2. N. F. Thornhill, M. A. A. Shoukat Choudhury and S. L. Shah, "The impact of compression
   on data-driven process analyses," *Journal of Process Control* **14**, 389–398 (2004),
   [doi:10.1016/j.jprocont.2003.06.003](https://doi.org/10.1016/j.jprocont.2003.06.003) —
   what Part C finds on one tag, measured across loop benchmarking, fault detection and data
   reconciliation, with the conclusion that compressed archives should be used with caution.
3. N. F. Thornhill, B. Huang and H. Zhang, "Detection of multiple oscillations in control
   loops," *Journal of Process Control* **13**, 91–100 (2003),
   [doi:10.1016/S0959-1524(02)00007-0](https://doi.org/10.1016/S0959-1524(02)00007-0) — the
   ACF as a working tool on plant historian data.